In [4]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 API Key
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从本地 scraper 模块导入 fetch_website_contents：抓取网页正文文本
from scraper import fetch_website_contents
# 从 IPython.display 导入展示工具：在笔记本里用 Markdown 漂亮地显示（本格未直接调用，留给扩展）
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI


In [5]:
# 创建 OpenAI 客户端：默认从环境变量 OPENAI_API_KEY 读取密钥（需事先在环境或 .env 中配置）
openai = OpenAI()


In [ ]:
# ========== Prompt + 抓取 + 调用模型：国会股票交易网页摘要 ==========

# system prompt：设定模型角色为「分析国会交易的个人理财助手」（发给模型的指令，保留英文原文）
system_prompt = "You are my personal financial assistant that analyzes congress trading within a website"
# user prompt：要求用表格简短总结近一个月交易，最近交易排前（保留英文；改译会改变模型行为）
user_prompt = """
Here are the contents of a website.
Provides a short summary in table form. 
I want the data from the past one month, 
the table should show who bought what company, show the recent trades first"
"""

# 第 2 步：组装 Chat Completions 所需的 messages 列表（system 定角色，user 放任务+网页正文）
def messages_for(website):
    # 返回两条消息；user 的 content = 固定前缀 + 抓到的网页文本
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + website}
    ] # fill this in

# 抓取国会交易网站正文（URL 保持原样，这是数据来源）
website = fetch_website_contents("https://www.capitoltrades.com/trades")

# 第三步：调用 OpenAI Chat Completions；model id 与 messages 逻辑保持原样
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages_for(website))
# 打印完整响应对象（含 choices、usage 等；纯文本摘要在 choices[0].message.content）
print(response)

# 第四步：打印结果
# 打印（响应.选择[0].消息.内容）


Here is a summary table of US politician stock trades from the past one month, showing the most recent trades first:

| Politician    | Party      | Position | Company             | Action | Size     | Price   | Date       | Owner      |
|---------------|------------|----------|---------------------|--------|----------|---------|------------|------------|
| Byron Donalds | Republican | House    | Brown & Brown Inc   | Sell   | 1K–15K   | $67.61  | 10 Feb 2026| Undisclosed|
| Byron Donalds | Republican | House    | Brown & Brown Inc   | Sell   | 1K–15K   | $67.61  | 10 Feb 2026| Spouse     |
| Byron Donalds | Republican | House    | PayPal Holdings Inc | Sell   | 1K–15K   | $41.49  | 10 Feb 2026| Spouse     |
| Byron Donalds | Republican | House    | PayPal Holdings Inc | Sell   | 1K–15K   | $41.49  | 10 Feb 2026| Undisclosed|
| Byron Donalds | Republican | House    | ServiceNow Inc      | Buy    | 1K–15K   | $106.48 | 10 Feb 2026| Undisclosed|
| Byron Donalds | Republican | House    | 